In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

plt_columns = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent',
    'hot', 'num_failed_logins', 'logged_in', 'num_compromised',
    'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds',
    'is_host_login', 'is_guest_login', 'count', 'srv_count',
    'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

train_df = pd.read_csv('data/raw/KDDTrain+.txt', header=None, names=plt_columns)

print("== 데이터 로딩 완료 ===")
print(f"shape: {train_df.shape}")

== 데이터 로딩 완료 ===
shape: (125973, 43)


In [3]:
# 1. attack type 타입 생성
attack_map = {
    'normal': 'Normal',
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS',
    'smurf': 'DoS', 'teardrop': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L',
    'phf': 'R2L', 'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R'
}

train_df['attack_type'] = train_df['label'].map(attack_map)
print("=== attack_type 생성 완료 ===")
print(train_df['attack_type'].value_counts())

=== attack_type 생성 완료 ===
attack_type
Normal    67343
DoS       45927
Probe     11656
R2L         995
U2R          52
Name: count, dtype: int64


In [4]:
#3. difficulty 컬럼 제거
train_df = train_df.drop(columns=['difficulty'])
print("=== difficulty 제거 완료 ===")
print(f"shape: {train_df.shape}")

=== difficulty 제거 완료 ===
shape: (125973, 43)


In [5]:
#4. 범주형 피처 인코딩
categorical_cols = ['protocol_type', 'service', 'flag']

# 인코더 저장용 딕셔너리 
encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    encoders[col] = le
    print(f"{col} 인코딩 완료 — 클래스 수: {len(le.classes_)}")

train_df[categorical_cols].head()

protocol_type 인코딩 완료 — 클래스 수: 3
service 인코딩 완료 — 클래스 수: 70
flag 인코딩 완료 — 클래스 수: 11


,protocol_type,service,flag
0,1,20,9
1,2,44,9
2,1,49,5
3,1,24,9
4,1,24,9


In [6]:
#5. 타겟 변수 인코딩
target_encoder = LabelEncoder()
train_df['attack_type_encoded'] = target_encoder.fit_transform(train_df['attack_type'])

# 매핑 확인
for i, cls in enumerate(target_encoder.classes_):
    print(f"{cls} → {i}")

DoS → 0
Normal → 1
Probe → 2
R2L → 3
U2R → 4


In [7]:
#6. 피처(X)와 타겟(y) 분리
# 학습에 사용할 피처만 선택 (label, attack_type 문자열 컬럼 제외)
feature_cols = [col for col in train_df.columns 
                 if col not in ['label', 'attack_type', 'attack_type_encoded']]

X = train_df[feature_cols]
y = train_df['attack_type_encoded']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print()
print("피처 목록:", feature_cols)

X shape: (125973, 41)
y shape: (125973,)

피처 목록: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate']


In [8]:
#7. Train / Validation / Test 분리 (7:1.5:1.5)
# 먼저 Train(70%) vs Temp(30%) 분리
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Temp를 Validation(15%) vs Test(15%)로 분리
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X_test.shape}")

Train: (88181, 41)
Val:   (18896, 41)
Test:  (18896, 41)


In [9]:
#8. StandardScaler 적용 (스케일링)
scaler = StandardScaler()

# train에만 fit_transform (학습 + 변환) test 데이터의 정보가 학습 과정에 섞이면 데이터 누수 발생 (아까 difficulty 때 배운 것과 같은 개념!)
X_train_scaled = scaler.fit_transform(X_train)

# val, test는 transform만 (train 기준으로만 변환)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# numpy array → DataFrame 다시 변환 (컬럼명 유지)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_val_scaled = pd.DataFrame(X_val_scaled, columns=X_val.columns, index=X_val.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print("=== 스케일링 완료 ===")
X_train_scaled.describe().loc[['mean', 'std']]

=== 스케일링 완료 ===


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate
mean,3.327861e-17,-1.284409e-16,-1.611555e-19,1.256207e-16,-2.417333e-19,-1.490689e-18,4.512354e-18,-5.156977e-18,-8.057776e-19,-4.520412e-17,...,4.947474e-17,4.721857e-17,6.865225e-17,4.125581e-17,-1.103915e-17,4.141697e-17,-1.684075e-17,-6.559030e-17,7.356749e-17,-7.066669e-17
std,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,...,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00,1.000006e+00


In [10]:
#9. SMOTE 적용 (Train)
print("=== SMOTE 적용 전 ===")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print()
print("=== SMOTE 적용 후 ===")
print(y_train_resampled.value_counts())

=== SMOTE 적용 전 ===
attack_type_encoded
1    47140
0    32149
2     8159
3      697
4       36
Name: count, dtype: int64

=== SMOTE 적용 후 ===
attack_type_encoded
2    47140
0    47140
1    47140
4    47140
3    47140
Name: count, dtype: int64


In [15]:
#10. 전처리 결과 저장
import pickle
import os

# 디렉토리 생성
os.makedirs('data/processed', exist_ok=True)
os.makedirs('models', exist_ok=True)

# 전처리된 데이터 저장
X_train_resampled.to_csv('data/processed/X_train.csv', index=False)
y_train_resampled.to_csv('data/processed/y_train.csv', index=False)
X_val_scaled.to_csv('data/processed/X_val.csv', index=False)
y_val.to_csv('data/processed/y_val.csv', index=False)
X_test_scaled.to_csv('data/processed/X_test.csv', index=False)
y_test.to_csv('data/processed/y_test.csv', index=False)

# 인코더 & 스케일러 저장 (나중에 새 데이터 변환할 때 재사용)
with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)

with open('models/target_encoder.pkl', 'wb') as f:
    pickle.dump(target_encoder, f)

with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("=== 전처리 결과 저장 완료 ===")
print(f"최종 학습 데이터 shape: {X_train_resampled.shape}")

=== 전처리 결과 저장 완료 ===
최종 학습 데이터 shape: (235700, 41)
